# 02a — Whisper Transcription (Colab GPU)

Generates ASR transcripts for the IEMOCAP utterances using **Whisper large-v3-turbo**,
and saves them to `transcripts.csv` for the EDA / modeling notebooks to load.

**This is a run-once, cache-it step.** It writes to Google Drive after every chunk, so a
dropped Colab session just resumes where it left off.

### Before you run
1. **Set a GPU runtime:** Runtime → Change runtime type → **T4 GPU**.
2. The first `load_dataset(...)` call downloads the dataset (~1.4 GB) — one time.
3. If the dataset is gated and asks for auth, run in a cell:
   `from huggingface_hub import login; login()` and paste a token from huggingface.co/settings/tokens.

Expected wall-clock: roughly **1.5–3 hours** for all ~10k clips on a free T4 (turbo runs several × faster than real-time). The checkpointing means you can stop and resume.


In [ ]:
%pip install -q -U transformers accelerate datasets jiwer

## 1. Mount Drive & set paths
Transcripts are written here so they survive a session restart.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/ser_project"   # <-- change if you like
os.makedirs(PROJECT_DIR, exist_ok=True)
OUT_CSV = os.path.join(PROJECT_DIR, "transcripts.csv")
print("Transcripts ->", OUT_CSV)

## 2. Imports & device check

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Audio
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1
print("CUDA available:", torch.cuda.is_available())
if DEVICE == -1:
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

## 3. Load the dataset

Whisper expects **16 kHz mono**, so we cast the `audio` column — decoded arrays then come
out at 16 kHz automatically.

In [ ]:
ds = load_dataset("AbstractTTS/IEMOCAP", split="train")
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

print(ds)
print("Columns:", ds.column_names)
print("Rows:", len(ds))
print("Example file :", ds[0]["file"])
print("Gold transcript:", ds[0]["transcription"])
print("Major emotion :", ds[0]["major_emotion"])

## 4. Build the Whisper pipeline

`large-v3-turbo`: near-large-v3 quality, much faster, transcription-only (we don't need
translation). `chunk_length_s=30` lets it handle the handful of clips longer than 30 s.
Language is forced to English and temperature pinned to 0 for reproducibility.

In [ ]:
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16 if DEVICE == 0 else torch.float32,
    device=DEVICE,
    chunk_length_s=30,
)
GEN_KWARGS = {"language": "en", "task": "transcribe", "temperature": 0.0}

## 5. Transcribe (with resume + checkpointing)

We transcribe **all** ~10k clips — filtering (dropping `xxx`, the 4-class ablation, etc.)
happens downstream in the EDA / modeling notebooks, not here. Output is keyed by `file`
(the wav basename, e.g. `Ses01F_impro01_F000.wav`), which is also how you'll join it back
to your `iemocap_full_dataset.csv` (`file` ↔ `os.path.basename(path)`).

If the session dies, just re-run this cell — it skips everything already in the CSV.

In [ ]:
BATCH_SIZE = 16    # ASR inference batch (lower to 8 if you hit OOM)
CHUNK = 200        # clips decoded + written per checkpoint

# --- resume: skip files already transcribed ---
done = set()
if os.path.exists(OUT_CSV):
    try:
        done = set(pd.read_csv(OUT_CSV, on_bad_lines="skip")["file"])
    except Exception as e:
        print("Could not read existing CSV, starting fresh:", e)
print("Already done:", len(done))

all_files = ds["file"]
todo = [i for i, f in enumerate(all_files) if f not in done]
print("Remaining:", len(todo))

write_header = not os.path.exists(OUT_CSV)
for start in range(0, len(todo), CHUNK):
    idx = todo[start:start + CHUNK]
    sub = ds.select(idx)
    arrays = [np.asarray(a["array"], dtype=np.float32) for a in sub["audio"]]
    outs = asr(arrays, batch_size=BATCH_SIZE, generate_kwargs=GEN_KWARGS)
    texts = [o["text"].strip() for o in outs]

    pd.DataFrame({
        "file": sub["file"],
        "major_emotion": sub["major_emotion"],
        "transcription": sub["transcription"],   # gold, kept for the WER check below
        "whisper_text": texts,
    }).to_csv(OUT_CSV, mode="a", header=write_header, index=False)
    write_header = False
    print(f"  saved {start + len(idx)} / {len(todo)}")

print("Done.")

## 6. Preview the results

In [ ]:
res = (pd.read_csv(OUT_CSV, on_bad_lines="skip")
         .drop_duplicates(subset="file", keep="last"))
print("Total transcribed:", len(res))
res.sample(min(8, len(res)))[["file", "major_emotion", "transcription", "whisper_text"]]

## 7. (Optional) WER sanity check vs the gold transcription

A quick confirmation that Whisper is working — and a preview of an analysis worth expanding
in the EDA notebook: *does transcription error rate rise with emotional intensity?*
(Angry / excited / crying speech is harder to transcribe.) Expand this per emotion, gender,
and session there.

In [ ]:
import jiwer

def normalize(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)   # strip punctuation
    s = re.sub(r"\s+", " ", s).strip()
    return s

valid = res.dropna(subset=["transcription", "whisper_text"]).copy()
valid["ref"] = valid["transcription"].map(normalize)
valid["hyp"] = valid["whisper_text"].map(normalize)
valid = valid[(valid["ref"].str.len() > 0) & (valid["hyp"].str.len() > 0)]

overall = jiwer.wer(list(valid["ref"]), list(valid["hyp"]))
print(f"Overall WER vs gold: {overall:.3f}  (n={len(valid)})")
print("\nPer-emotion WER:")
for emo, g in valid.groupby("major_emotion"):
    w = jiwer.wer(list(g["ref"]), list(g["hyp"]))
    print(f"  {emo:12s} WER={w:.3f}  n={len(g)}")

## 8. Next steps

- **Download** `transcripts.csv` from Drive (or just read it from Drive in your EDA notebook).
- **Join** into your metadata: `meta["file"] = meta["path"].map(os.path.basename)`, then merge
  on `file`. That brings session (for leave-session-out CV), agreement (for the `xxx` drop),
  and method alongside the Whisper text.
- In **`02_eda.ipynb`**, add the text layer: word/token counts and speaking rate per emotion,
  distinctive vocabulary, sentiment-vs-valence sanity checks, and the per-emotion/gender/session
  WER analysis previewed above.

Fit any text feature that involves learning (TF-IDF vocab, etc.) on **training folds only**
under leave-session-out CV — same leakage rule as the acoustic features.